# Parte 2 · La geometría de una predicción (2.1 y 2.2)

**Autor:** Herman Echeverría · Grupo Taiwan

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

Datos: Taiwan Credit, estandarizados como en 0.1, con los hiperparámetros de la validación cruzada (`resultados/hiperparametros.json`). La ficha de evidencia de cada predicción se resume con

$$N_{\text{ef}}(x) = \frac{\left(\sum_i w_i(x)\right)^2}{\sum_i w_i(x)^2}, \qquad c(x) = \frac{\max_i w_i(x)^2}{\sum_j w_j(x)^2},$$

calculados con `src/evidencia.py`.

In [ ]:
import sys, json
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from datos import cargar_taiwan
from pesos import pesos_ols, pesos_ridge, pesos_knn, pesos_arbol
from bosque import pesos_bosque, verificar
from evidencia import n_efectivo, concentracion, testigos

X_train, X_test, y_train, y_test, sd = cargar_taiwan("../data/taiwan_credit.csv")
X_train_crudo, X_test_crudo, _, _, _ = cargar_taiwan("../data/taiwan_credit.csv", escalar=False)  # para leer los testigos
n = len(X_train)

hp = json.load(open("../resultados/hiperparametros.json"))
hp

In [ ]:
modelos = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=hp["lambda"]),
    "k-NN": KNeighborsRegressor(n_neighbors=hp["k"]),
    "Árbol": DecisionTreeRegressor(max_depth=hp["max_depth"], random_state=0),
    "Bosque": RandomForestRegressor(n_estimators=200, min_samples_leaf=hp["min_samples_leaf"],
                                    max_features=0.33, bootstrap=False, random_state=0, n_jobs=-1),
}
for m in modelos.values():
    m.fit(X_train, y_train)


def pesos(nombre, X_q):
    if nombre == "OLS":
        return pesos_ols(X_train, X_q)
    if nombre == "Ridge":
        return pesos_ridge(X_train, X_q, hp["lambda"])
    if nombre == "k-NN":
        return pesos_knn(modelos["k-NN"], X_q, n)
    if nombre == "Árbol":
        return pesos_arbol(modelos["Árbol"], X_train, X_q)
    return pesos_bosque(modelos["Bosque"], X_train, X_q)

## 2.1 · $N_{\text{ef}}$ del bosque en todo el conjunto de prueba

La matriz completa de pesos serían $6\,000 \times 24\,000$ números (~1.2 GB), así que se calcula por lotes de 500 consultas y de cada lote solo se guardan $N_{\text{ef}}$ y $c(x)$. En cada lote se verifica además que los pesos reproducen el `.predict()`.

In [ ]:
lote = 500
nef_bosque, c_bosque = np.empty(len(X_test)), np.empty(len(X_test))
for inicio in range(0, len(X_test), lote):
    X_q = X_test.iloc[inicio:inicio + lote]
    W = pesos_bosque(modelos["Bosque"], X_train, X_q)
    verificar(W, y_train, modelos["Bosque"].predict(X_q))
    nef_bosque[inicio:inicio + lote] = n_efectivo(W)
    c_bosque[inicio:inicio + lote] = concentracion(W)

pd.DataFrame({"N_ef del bosque": nef_bosque, "c(x) del bosque": c_bosque}).describe().round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(nef_bosque, bins=60)
ax.axvline(np.median(nef_bosque), color="tab:red", ls="--", label=f"mediana = {np.median(nef_bosque):.0f}")
ax.set_xlabel(r"$N_{ef}(x)$ del bosque"); ax.set_ylabel("filas de prueba")
ax.set_title("¿En cuántas filas se apoya cada predicción del bosque?")
ax.legend(); fig.tight_layout()
fig.savefig("../figs/parte2_nef_bosque.png", dpi=150)
plt.show()

### Las tres consultas

1. La de $N_{\text{ef}}$ **mediano**: una predicción típica.
2. La de $N_{\text{ef}}$ **mínimo**: la que se apoya en menos filas.
3. La de **mayor desacuerdo entre los cinco modelos**: la fila de prueba donde la diferencia entre la predicción más alta y la más baja es máxima.

In [ ]:
pred_test = pd.DataFrame({nombre: m.predict(X_test) for nombre, m in modelos.items()})
desacuerdo = (pred_test.max(axis=1) - pred_test.min(axis=1)).to_numpy()

consultas = {
    "nef_mediano": int(np.argmin(np.abs(nef_bosque - np.median(nef_bosque)))),
    "nef_minimo": int(np.argmin(nef_bosque)),
    "mayor_desacuerdo": int(np.argmax(desacuerdo)),
}
# posición en X_test (iloc) e índice original del csv, para que 4.2 use las mismas filas
with open("../resultados/consultas.json", "w") as f:
    json.dump({k: {"posicion": p, "indice": int(X_test.index[p])} for k, p in consultas.items()}, f, indent=2)

pd.DataFrame({k: {"posición en prueba": p, "índice original": X_test.index[p], "y real": y_test[p],
                  "N_ef bosque": nef_bosque[p], "desacuerdo": desacuerdo[p]}
              for k, p in consultas.items()}).T.round(3)

In [ ]:
filas = []
W_consultas = {}
for consulta, p in consultas.items():
    X_q = X_test.iloc[[p]]
    for nombre, m in modelos.items():
        w = pesos(nombre, X_q)[0]
        verificar(w, y_train, m.predict(X_q)[0])
        W_consultas[consulta, nombre] = w
        filas.append({"consulta": consulta, "modelo": nombre, "predicción": m.predict(X_q)[0],
                      "N_ef": n_efectivo(w)[0], "c(x)": concentracion(w)[0],
                      "pesos ≠ 0": int((w != 0).sum()), "pesos < 0": int((w < 0).sum())})
ficha = pd.DataFrame(filas).set_index(["consulta", "modelo"])
ficha.round(4)

### Los testigos

Los 5 testigos de cada consulta y modelo son las filas de entrenamiento con mayor $|w_i|$. Se muestran su peso, su $y$ y sus variables más importantes en escala original: las 4 de mayor importancia en el bosque, más el límite de crédito y la edad para reconocer al cliente. `PAY_0` a `PAY_6` son los meses de atraso en el pago (−2 a −1 = pagó a tiempo, 1 o más = meses de retraso).

In [ ]:
importancia = pd.Series(modelos["Bosque"].feature_importances_, index=X_train.columns).sort_values(ascending=False)
variables = list(dict.fromkeys(list(importancia.index[:4]) + ["LIMIT_BAL", "AGE"]))
importancia.head(6).round(3).to_frame("importancia en el bosque")

In [ ]:
def tabla_testigos(consulta):
    p = consultas[consulta]
    partes = [X_test_crudo.iloc[[p]][variables].assign(modelo="(consulta)", w=np.nan, y=y_test[p])]
    for nombre in modelos:
        w = W_consultas[consulta, nombre]
        idx = testigos(w, k=5)
        partes.append(X_train_crudo.iloc[idx][variables].assign(modelo=nombre, w=w[idx], y=y_train[idx]))
    return pd.concat(partes)[["modelo", "w", "y"] + variables].round(4)

tabla_testigos("nef_mediano")

In [ ]:
tabla_testigos("nef_minimo")

In [ ]:
tabla_testigos("mayor_desacuerdo")

**Lectura de las tres fichas.**

En todo el conjunto de prueba, el $N_{\text{ef}}$ del bosque va de 128 a 2 371 filas, con mediana 752. Con `min_samples_leaf = 50` cada árbol promedia hojas grandes, así que ninguna predicción se apoya en un puñado de clientes. $c(x)$ tiene mediana 0.03 y máximo 0.16: en ninguna consulta manda una sola fila.

1. **$N_{\text{ef}}$ mediano** (índice 591, no cayó en mora). Es un cliente que paga a tiempo (`PAY_0`–`PAY_4` = 0), con límite de 160 000 y 34 años. k-NN y el bosque predicen 6 % de mora y sus testigos se le parecen mucho: también pagan a tiempo, con límites de 130 000 a 190 000. El árbol predice 10 % con $N_{\text{ef}} = 11\,767$: a profundidad 4 mete en la misma hoja a casi la mitad del entrenamiento (todos los que pagan a tiempo), así que su "evidencia" es enorme pero poco específica. OLS predice 24 % con $N_{\text{ef}} \approx 4\,400$, aunque 7 706 de sus pesos son negativos. Su testigo principal tiene peso **negativo** y no se parece a la consulta (límite de 340 000 y un pago de 873 552). En OLS, un $N_{\text{ef}}$ grande no significa "muchos clientes parecidos".

2. **$N_{\text{ef}}$ mínimo** (índice 11705, no cayó en mora). Tiene 2 a 3 meses de atraso, un límite de solo 20 000 y 25 años. Es un perfil poco común, por eso el bosque solo reúne unas 128 filas efectivas. Sus 5 testigos son casi copias de la consulta (atraso de 2–3 meses, límites de 10 000–30 000, 22–26 años) y los 5 cayeron en mora, así que el bosque predice 77 %. Este cliente **no** cayó en mora: es justo el tipo de predicción con poca evidencia en la que conviene desconfiar. Los testigos de OLS vuelven a ser filas extremas (`PAY_3` de 7–8 meses), no vecinos.

3. **Mayor desacuerdo** (índice 13712, sí cayó en mora). Se escogió porque es donde los cinco modelos más difieren (0.75 entre la predicción más alta y la más baja). Que la geometría del modelo cambie tanto la respuesta indica que el dato no la decide por sí solo. Es un cliente con 8, 7, 6 y 5 meses de atraso, un caso extremo. OLS y ridge predicen **1.26–1.27**, que no es una probabilidad: extrapolan linealmente en los `PAY`. Los modelos locales dan 0.52–0.60 porque sus testigos, con el mismo patrón 8-7-6-5, en su mayoría **no** cayeron en mora en el entrenamiento. El bosque se apoya en apenas 181 filas efectivas.

**Lo que sale de las tres fichas:** en los modelos locales (k-NN, árbol y bosque) los testigos sí explican la predicción ("se parece a estos clientes, y tantos cayeron en mora"). En OLS y ridge los testigos son filas lejanas con mucho apalancamiento, y $N_{\text{ef}}$ mide qué tan repartido está un voto con signos mezclados, no cuánta evidencia local hay.